# Mechanism Bridge: Does amplifying pairing heads weaken the task vector?

**Claim:** Pairing heads write demo-specific (non-abstract) information into the residual stream.
Amplifying them should contaminate the task vector with binding info,
making it less patchable to new queries even if ICL accuracy stays high.

**Design:**
1. Classify Hendel prompts as "survives" (correct under shuffled inputs) vs "fails" (incorrect under shuffled inputs)
2. For each prompt, amplify pairing heads (2x, 2.5x, 3x) during the forward pass
3. Extract the task vector (residual at final position) and patch into a zero-shot prompt
4. Compare patching accuracy: normal vs OI-amplified vs random-amplified (control)

**Prediction:** OI amplification degrades patching more on "fails" prompts than "survives" prompts,
because those prompts relied on input-output binding — and amplifying that signal contaminates the abstract task vector.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import sys; sys.path.insert(0, '../..')
import torch, numpy as np, matplotlib.pyplot as plt, random, seaborn as sns
import pandas as pd, pickle
from collections import defaultdict
from tqdm import tqdm
from transformer_lens import HookedTransformer

try:
    from configs import MODEL_NAME
except ImportError:
    MODEL_NAME = 'meta-llama/Llama-3.2-3B'

%matplotlib inline
sns.set_context('notebook')

model = HookedTransformer.from_pretrained(MODEL_NAME, device='cuda', dtype=torch.float16)
model.eval()
n_layers = model.cfg.n_layers
n_heads  = model.cfg.n_heads
d_model  = model.cfg.d_model
PATTERN_FILTER = lambda name: 'attn.hook_pattern' in name
top10pct_n = max(1, int(0.10 * n_layers * n_heads))
all_pairs = [(L, h) for L in range(n_layers) for h in range(n_heads)]

## Utilities

In [ ]:
def build_icl_prompt(demo_pairs, query_input):
    lines = [f'{inp} \u2192 {out}' for inp, out in demo_pairs]
    lines.append(f'{query_input} \u2192')
    return '\n'.join(lines)

def find_per_demo_positions(model, prompt, demos):
    tokens = model.to_tokens(prompt, prepend_bos=True)[0]
    full_decoded = model.tokenizer.decode(tokens)
    char_to_tok = []
    for i in range(len(tokens)):
        s = model.tokenizer.decode([tokens[i].item()])
        char_to_tok.extend([i] * len(s))
    def span_to_toks(start, end):
        if start < 0 or end > len(char_to_tok): return []
        return sorted(set(char_to_tok[start:end]))
    result = []
    search_from = 0
    for inp, out in demos:
        demo = {'input_positions': [], 'output_positions': []}
        inp_s, out_s = str(inp), str(out)
        idx = full_decoded.find(inp_s, search_from)
        if idx >= 0:
            demo['input_positions'] = span_to_toks(idx, idx + len(inp_s))
            search_from = idx + len(inp_s)
        arrow = full_decoded.find('\u2192', search_from)
        if arrow >= 0:
            oidx = full_decoded.find(out_s, arrow)
            if oidx >= 0:
                demo['output_positions'] = span_to_toks(oidx, oidx + len(out_s))
                search_from = oidx + len(out_s)
        result.append(demo)
    return result

@torch.no_grad()
def check_correct(model, tokens, expected, max_new_tokens=20, hooks=None):
    """Multi-token greedy generation check."""
    target = expected.strip()
    generated = []
    cur = tokens.clone()
    for _ in range(max_new_tokens):
        if hooks:
            logits = model.run_with_hooks(cur, fwd_hooks=hooks)[0, -1]
        else:
            logits = model(cur)[0, -1]
        nt = logits.argmax().item()
        generated.append(nt)
        cur = torch.cat([cur, torch.tensor([[nt]], device=cur.device)], dim=1)
        decoded = model.tokenizer.decode(generated).strip()
        if decoded == target: return 1
        if len(decoded) >= len(target): return 0
        if decoded and not target.startswith(decoded): return 0
    return int(model.tokenizer.decode(generated).strip() == target)

@torch.no_grad()
def check_first_token(model, tokens, expected, hooks=None):
    """Single forward pass: does top logit match first token of expected?"""
    target_toks = model.to_tokens(' ' + expected.strip(), prepend_bos=False)[0]
    space_id = model.to_tokens(' ', prepend_bos=False)[0, 0].item()
    target_id = None
    for t in target_toks:
        if t.item() != space_id:
            target_id = t.item()
            break
    if target_id is None:
        target_id = target_toks[-1].item()
    if hooks:
        logits = model.run_with_hooks(tokens, fwd_hooks=hooks)[0, -1]
    else:
        logits = model(tokens)[0, -1]
    return int(logits.argmax().item() == target_id)

def heads_by_layer(head_list):
    d = defaultdict(list)
    for L, h in head_list: d[L].append(h)
    return d

def make_ablation_hooks(hbl):
    hooks = []
    for L, h_list in hbl.items():
        def hook(z, hook, _hl=h_list):
            for h in _hl: z[0, :, h, :] = 0
            return z
        hooks.append((f'blocks.{L}.attn.hook_z', hook))
    return hooks

def make_amplify_hooks(head_list, scale):
    hbl = defaultdict(list)
    for L, h in head_list: hbl[L].append(h)
    hooks = []
    for L, h_list in hbl.items():
        def hook(z, hook, _hl=h_list, _s=scale):
            for h in _hl:
                z[0, :, h, :] *= _s
            return z
        hooks.append((f'blocks.{L}.attn.hook_z', hook))
    return hooks

def score_oi_heads(prompts_data, desc='OI Scoring'):
    num_sum  = np.zeros((n_layers, n_heads), dtype=np.float64)
    base_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    cnt = 0
    for pdata in tqdm(prompts_data, desc=desc, leave=False):
        demos = pdata['demo_pairs']
        tokens = model.to_tokens(pdata['prompt'], prepend_bos=True)
        seq_len = tokens.shape[1]
        per_demo = find_per_demo_positions(model, pdata['prompt'], demos)
        all_in = set(); all_out = set()
        for d in per_demo:
            for p in d.get('input_positions', []):  all_in.add(p)
            for p in d.get('output_positions', []): all_out.add(p)
        num_mask = torch.zeros(seq_len, seq_len)
        for d in per_demo:
            for op in d.get('output_positions', []):
                for ip in d.get('input_positions', []):
                    num_mask[op, ip] = 1.0
        base_mask = torch.zeros(seq_len, seq_len)
        ins_list = sorted(all_in)
        for q in range(seq_len):
            if q in all_out: continue
            for ik in ins_list:
                if ik <= q: base_mask[q, ik] = 1.0
        n_num = int(num_mask.sum().item()); n_base = int(base_mask.sum().item())
        if n_num == 0 or n_base == 0: continue
        with torch.no_grad():
            _, cache = model.run_with_cache(tokens, names_filter=PATTERN_FILTER)
        for L in range(n_layers):
            patt = cache['pattern', L][0].cpu().float()
            num_sum[L, :]  += (patt * num_mask).sum(dim=(1, 2)).numpy() / n_num
            base_sum[L, :] += (patt * base_mask).sum(dim=(1, 2)).numpy() / n_base
        cnt += 1
        del cache; torch.cuda.empty_cache()
    return (num_sum - base_sum) / max(cnt, 1)

## Load Hendel tasks

In [ ]:
with open('../../configs/splits.pkl', 'rb') as f:
    splits = pickle.load(f)

HENDEL_PROMPTS = {}
for task_key in splits:
    HENDEL_PROMPTS[task_key] = splits[task_key]['icl_prompts']

HENDEL_TASKS = list(HENDEL_PROMPTS.keys())
print(f'{len(HENDEL_TASKS)} Hendel tasks:')
for t in HENDEL_TASKS:
    print(f'  {t:<45s} n={len(HENDEL_PROMPTS[t])}')

## Classify prompts: survives vs fails under shuffled inputs

In [ ]:
hendel_classification = {}

for task_name in HENDEL_TASKS:
    results = []
    for pdata in tqdm(HENDEL_PROMPTS[task_name], desc=f'{task_name} classify'):
        tokens = model.to_tokens(pdata['prompt'], prepend_bos=True)
        ans = pdata['query_output']
        orig_correct = check_correct(model, tokens, ans)
        
        demo_pairs = pdata['demo_pairs']
        inputs_shuffled = [x for x, y in demo_pairs]
        random.shuffle(inputs_shuffled)
        shuffled_pairs = list(zip(inputs_shuffled, [y for x, y in demo_pairs]))
        shuffled_prompt = build_icl_prompt(shuffled_pairs, pdata['query_input'])
        shuffled_tokens = model.to_tokens(shuffled_prompt, prepend_bos=True)
        shuffle_correct = check_correct(model, shuffled_tokens, ans)
        
        results.append({
            'pdata': pdata,
            'orig_correct': orig_correct,
            'shuffle_correct': shuffle_correct,
        })
        torch.cuda.empty_cache()
    
    hendel_classification[task_name] = results
    n_orig = sum(r['orig_correct'] for r in results)
    n_surv = sum(r['orig_correct'] and r['shuffle_correct'] for r in results)
    n_fail = sum(r['orig_correct'] and not r['shuffle_correct'] for r in results)
    print(f'  {task_name:<45s} orig={n_orig}/{len(results)}  survives={n_surv}  fails={n_fail}')

## Score and select OI heads on Hendel tasks

In [ ]:
all_hendel_prompts = []
for task_name in HENDEL_TASKS:
    for r in hendel_classification[task_name]:
        if r['orig_correct']:
            all_hendel_prompts.append(r['pdata'])

print(f'Scoring OI heads on {len(all_hendel_prompts)} correct Hendel prompts...')
rng_sub = random.Random(42)
scoring_sample = rng_sub.sample(all_hendel_prompts, min(100, len(all_hendel_prompts)))
hendel_oi_scores = score_oi_heads(scoring_sample, desc='Hendel OI')

flat = sorted([(L, h, hendel_oi_scores[L, h]) for L in range(n_layers) for h in range(n_heads)],
              key=lambda x: -x[2])
hendel_oi_top = [(L, h) for L, h, _ in flat[:top10pct_n]]
print(f'Top {top10pct_n} OI heads selected')
print('Top 10:', [(L, h, f'{s:.3f}') for L, h, s in flat[:10]])

## Amplification experiment

For each correct Hendel prompt:
- Run normal, OI amplified (2x, 2.5x, 3x), and random amplified (2x, 2.5x, 3x)
- Measure ICL accuracy (multi-token) and task vector quality (patching, first-token)

In [ ]:
PATCH_LAYERS_AMP = list(range(n_layers))
N_PER_TASK_AMP = 50

oi_set_amp = set(hendel_oi_top)
rng_amp = random.Random(888)
rand_heads_amp = rng_amp.sample([p for p in all_pairs if p not in oi_set_amp], top10pct_n)

CONDITIONS = [
    ('normal', []),
    ('oi_2x', make_amplify_hooks(hendel_oi_top, 2.0)),
    ('oi_2.5x', make_amplify_hooks(hendel_oi_top, 2.5)),
    ('oi_3x', make_amplify_hooks(hendel_oi_top, 3.0)),
    ('rand_2x', make_amplify_hooks(rand_heads_amp, 2.0)),
    ('rand_2.5x', make_amplify_hooks(rand_heads_amp, 2.5)),
    ('rand_3x', make_amplify_hooks(rand_heads_amp, 3.0)),
]

amp_results = []

for task_name in tqdm(HENDEL_TASKS, desc='Tasks'):
    classified = hendel_classification[task_name]
    correct = [r for r in classified if r['orig_correct']]
    
    for r in tqdm(correct[:N_PER_TASK_AMP], desc=f'{task_name}', leave=False):
        pdata = r['pdata']
        group = 'survives' if r['shuffle_correct'] else 'fails'
        tokens_icl = model.to_tokens(pdata['prompt'], prepend_bos=True)
        ans = pdata['query_output']
        
        other = [r2 for r2 in classified if r2['pdata']['query_input'] != pdata['query_input']]
        if not other: continue
        zs_pdata = other[0]['pdata']
        zs_prompt = f"{zs_pdata['query_input']} \u2192"
        zs_answer = zs_pdata['query_output']
        tokens_zs = model.to_tokens(zs_prompt, prepend_bos=True)
        
        row = {'task': task_name, 'group': group}
        
        for condition, hooks_list in CONDITIONS:
            # ICL accuracy (multi-token)
            if hooks_list:
                icl_acc = check_correct(model, tokens_icl, ans, hooks=hooks_list)
            else:
                icl_acc = check_correct(model, tokens_icl, ans)
            
            # Extract task vector
            with torch.no_grad():
                if hooks_list:
                    with model.hooks(fwd_hooks=hooks_list):
                        _, cache = model.run_with_cache(tokens_icl,
                            names_filter=lambda n: 'resid_post' in n)
                else:
                    _, cache = model.run_with_cache(tokens_icl,
                        names_filter=lambda n: 'resid_post' in n)
            
            # Patch across layers (single forward pass per layer)
            layer_accs = []
            for L in PATCH_LAYERS_AMP:
                resid = cache['resid_post', L][0, -1].clone()
                def make_patch(vec):
                    def hook(r, hook, _v=vec):
                        r[0, -1] = _v
                        return r
                    return hook
                pacc = check_first_token(model, tokens_zs, zs_answer,
                    hooks=[(f'blocks.{L}.hook_resid_post', make_patch(resid))])
                layer_accs.append(pacc)
            
            row[f'{condition}_icl'] = icl_acc
            row[f'{condition}_patch'] = max(layer_accs)
            row[f'{condition}_curve'] = layer_accs
            
            del cache; torch.cuda.empty_cache()
        
        amp_results.append(row)
    
    sub = [r for r in amp_results if r['task'] == task_name]
    print(f'{task_name}')
    print(f'  ICL:   norm={np.mean([r["normal_icl"] for r in sub]):.2f}  '
          f'oi2x={np.mean([r["oi_2x_icl"] for r in sub]):.2f}  '
          f'oi2.5x={np.mean([r["oi_2.5x_icl"] for r in sub]):.2f}  '
          f'oi3x={np.mean([r["oi_3x_icl"] for r in sub]):.2f}  '
          f'r2x={np.mean([r["rand_2x_icl"] for r in sub]):.2f}  '
          f'r2.5x={np.mean([r["rand_2.5x_icl"] for r in sub]):.2f}  '
          f'r3x={np.mean([r["rand_3x_icl"] for r in sub]):.2f}')
    print(f'  Patch: norm={np.mean([r["normal_patch"] for r in sub]):.2f}  '
          f'oi2x={np.mean([r["oi_2x_patch"] for r in sub]):.2f}  '
          f'oi2.5x={np.mean([r["oi_2.5x_patch"] for r in sub]):.2f}  '
          f'oi3x={np.mean([r["oi_3x_patch"] for r in sub]):.2f}  '
          f'r2x={np.mean([r["rand_2x_patch"] for r in sub]):.2f}  '
          f'r2.5x={np.mean([r["rand_2.5x_patch"] for r in sub]):.2f}  '
          f'r3x={np.mean([r["rand_3x_patch"] for r in sub]):.2f}')

df_amp = pd.DataFrame(amp_results)

## Aggregate plots (all conditions, with error bars)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

conditions = ['normal', 'oi_2x', 'oi_2.5x', 'oi_3x', 'rand_2x', 'rand_2.5x', 'rand_3x']
labels = ['Normal', 'OI 2x', 'OI 2.5x', 'OI 3x', 'Rand 2x', 'Rand 2.5x', 'Rand 3x']
colors = ['tab:blue', 'salmon', '#cc3333', 'tab:red', 'silver', '#aaaaaa', 'gray']

# Panel 1: ICL accuracy
ax = axes[0]
plot_rows = []
for _, row in df_amp.iterrows():
    for c, l in zip(conditions, labels):
        plot_rows.append({'Condition': l, 'Accuracy': row[f'{c}_icl']})
pdf = pd.DataFrame(plot_rows)
sns.barplot(data=pdf, x='Condition', y='Accuracy', order=labels,
            palette=dict(zip(labels, colors)), errorbar=('ci', 95), ax=ax)
ax.set_ylim(0, 1.05)
ax.set_title('ICL accuracy (should stay high)')
ax.set_xlabel('')
ax.grid(True, alpha=0.3, axis='y')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)

# Panel 2: Peak patching accuracy
ax = axes[1]
plot_rows2 = []
for _, row in df_amp.iterrows():
    for c, l in zip(conditions, labels):
        plot_rows2.append({'Condition': l, 'Patching': row[f'{c}_patch']})
pdf2 = pd.DataFrame(plot_rows2)
sns.barplot(data=pdf2, x='Condition', y='Patching', order=labels,
            palette=dict(zip(labels, colors)), errorbar=('ci', 95), ax=ax)
ax.set_ylim(0, 1.05)
ax.set_title('Peak patching accuracy')
ax.set_xlabel('')
ax.grid(True, alpha=0.3, axis='y')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)

# Panel 3: Patching curves for normal vs OI 2x vs rand 2x
ax = axes[2]
for cond, label, color, ls in [
    ('normal', 'Normal', 'tab:blue', '-'),
    ('oi_2x', 'OI 2x', 'tab:red', '--'),
    ('rand_2x', 'Rand 2x', 'gray', ':'),
]:
    curves = np.array([r[f'{cond}_curve'] for r in amp_results])
    ax.plot(PATCH_LAYERS_AMP, curves.mean(axis=0), color=color, lw=2, ls=ls, label=label)
ax.set_xlabel('Layer')
ax.set_ylabel('Patching accuracy')
ax.set_title('Per-layer patching curves')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Key analysis: filtered by ICL correctness, split by survive/fail

In [ ]:
# Filter to prompts correct under normal, oi_2x, and rand_2x
clean = df_amp[(df_amp['normal_icl'] == 1) &
               (df_amp['oi_2x_icl'] == 1) &
               (df_amp['rand_2x_icl'] == 1)]

print(f'Prompts correct under all 2x conditions: {len(clean)}/{len(df_amp)}')
print()

# Unfiltered summary
print('=== UNFILTERED ===')
for grp in ['survives', 'fails']:
    sub = df_amp[df_amp['group'] == grp]
    if len(sub) == 0: continue
    nm = sub['normal_patch'].mean()
    oi = sub['oi_2x_patch'].mean()
    rd = sub['rand_2x_patch'].mean()
    print(f'{grp} (n={len(sub)}):  normal={nm:.3f}  oi_2x={oi:.3f}  rand_2x={rd:.3f}  '
          f'OI_drop={nm-oi:+.3f}  rand_drop={nm-rd:+.3f}  '
          f'OI_specific={rd-oi:+.3f}')

print()

# Filtered summary
print('=== FILTERED (correct under all conditions) ===')
for grp in ['survives', 'fails']:
    sub = clean[clean['group'] == grp]
    if len(sub) == 0: continue
    nm = sub['normal_patch'].mean()
    oi = sub['oi_2x_patch'].mean()
    rd = sub['rand_2x_patch'].mean()
    print(f'{grp} (n={len(sub)}):  normal={nm:.3f}  oi_2x={oi:.3f}  rand_2x={rd:.3f}  '
          f'OI_drop={nm-oi:+.3f}  rand_drop={nm-rd:+.3f}  '
          f'OI_specific={rd-oi:+.3f}')

# Also show 2.5x filtered
clean25 = df_amp[(df_amp['normal_icl'] == 1) &
                 (df_amp['oi_2.5x_icl'] == 1) &
                 (df_amp['rand_2.5x_icl'] == 1)]
print()
print('=== FILTERED 2.5x ===')
for grp in ['survives', 'fails']:
    sub = clean25[clean25['group'] == grp]
    if len(sub) == 0: continue
    nm = sub['normal_patch'].mean()
    oi = sub['oi_2.5x_patch'].mean()
    rd = sub['rand_2.5x_patch'].mean()
    print(f'{grp} (n={len(sub)}):  normal={nm:.3f}  oi_2.5x={oi:.3f}  rand_2.5x={rd:.3f}  '
          f'OI_drop={nm-oi:+.3f}  rand_drop={nm-rd:+.3f}  '
          f'OI_specific={rd-oi:+.3f}')

In [ ]:
# === Grouped bar plot: patching by group (filtered, 2x) ===

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel 1: Patching accuracy by group
ax = axes[0]
plot_rows = []
for _, row in clean.iterrows():
    plot_rows.append({'Group': row['group'], 'Condition': 'Normal', 'Peak': row['normal_patch']})
    plot_rows.append({'Group': row['group'], 'Condition': 'OI 2x', 'Peak': row['oi_2x_patch']})
    plot_rows.append({'Group': row['group'], 'Condition': 'Rand 2x', 'Peak': row['rand_2x_patch']})
pdf = pd.DataFrame(plot_rows)
sns.barplot(data=pdf, x='Group', y='Peak', hue='Condition',
            palette={'Normal': 'tab:blue', 'OI 2x': 'tab:red', 'Rand 2x': 'lightgray'},
            errorbar=('ci', 95), ax=ax)
ax.set_ylim(0, 1.05)
ax.set_title('Task vector quality under OI amplification (2x)\n(filtered: correct under all conditions)')
ax.set_ylabel('Peak patching accuracy')
ax.grid(True, alpha=0.3, axis='y')

# Panel 2: Specific effect (rand_2x - oi_2x) by group
ax = axes[1]
spec_rows = []
for _, row in clean.iterrows():
    spec_rows.append({'Group': row['group'],
                      'Specific effect': row['rand_2x_patch'] - row['oi_2x_patch']})
sdf = pd.DataFrame(spec_rows)
sns.barplot(data=sdf, x='Group', y='Specific effect', hue='Group',
            palette={'survives': 'tab:blue', 'fails': 'tab:red'},
            errorbar=('ci', 95), legend=False, ax=ax)
ax.axhline(0, color='black', lw=0.5)
ax.set_title('OI-specific patching degradation\n(rand_2x \u2212 oi_2x patching accuracy)')
ax.set_ylabel('Specific effect on patching')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout(); plt.show()

## Per-task breakdown

In [ ]:
print(f'{"Task":<45s} {"n_s":>4s} {"n_f":>4s} {"spec_surv":>10s} {"spec_fail":>10s} {"diff":>8s}')
print('-' * 85)
for task_name in HENDEL_TASKS:
    sub = clean[clean['task'] == task_name]
    surv = sub[sub['group'] == 'survives']
    fail = sub[sub['group'] == 'fails']
    if len(surv) == 0 or len(fail) == 0:
        print(f'{task_name:<45s} {len(surv):>4d} {len(fail):>4d}   (need both groups)')
        continue
    spec_s = surv['rand_2x_patch'].mean() - surv['oi_2x_patch'].mean()
    spec_f = fail['rand_2x_patch'].mean() - fail['oi_2x_patch'].mean()
    print(f'{task_name:<45s} {len(surv):>4d} {len(fail):>4d} {spec_s:>+10.3f} {spec_f:>+10.3f} {spec_f - spec_s:>+8.3f}')